# 16 · Contest Patterns

*Contest essentials* · 9 exercises

Two pointers, sliding windows, prefix sums, grids and greedy choices.

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ch-16) · [Answer key](https://bloomcode.github.io/cp/python/answer-key.html#ch-16)

**Save your work:** this notebook is read-only, so first use **File → Save a copy in Drive** and work in that copy.

**How to work:** run the setup cell once. For each exercise, replace the stub in its code cell, run the cell, then run the test cell under it. Later exercises often reuse earlier ones, so go in order. Stuck? Each exercise links to its tested solution in the answer key.

In [ ]:
# Setup: run once.
import random
from collections import deque

try:
    import pytest
except ImportError:  # the tests only need pytest.raises; a small stand-in keeps the notebook self-contained
    import re as _re

    class _Raises:
        def __init__(self, kind, match=None):
            self.kind = kind
            self.match = match
            self.value = None

        def __enter__(self):
            return self

        def __exit__(self, kind, value, tb):
            if kind is None:
                raise AssertionError(f"did not raise {self.kind.__name__}")
            if not issubclass(kind, self.kind):
                return False
            self.value = value
            if self.match is not None and not _re.search(self.match, str(value)):
                raise AssertionError(f"exception message {str(value)!r} did not match {self.match!r}")
            return True

    class pytest:  # noqa: N801
        raises = _Raises

def _check_cmp(l, r, op, ls, rs):
    __tracebackhide__ = True
    passed = False
    if op == "==": passed = (l == r)
    elif op == "!=": passed = (l != r)
    elif op == "<": passed = (l < r)
    elif op == "<=": passed = (l <= r)
    elif op == ">": passed = (l > r)
    elif op == ">=": passed = (l >= r)
    elif op == "is": passed = (l is r)
    elif op == "is not": passed = (l is not r)
    elif op == "in": passed = (l in r)
    elif op == "not in": passed = (l not in r)
    if not passed:
        import inspect
        caller = inspect.currentframe().f_back
        locs = {k: v for k, v in caller.f_locals.items()
                if not k.startswith("_") and type(v) in (int, float, str, bool, list, tuple, dict, set)}
        lines = [
            f"Assertion Failed: {ls} {op} {rs}",
            f"  Expected: {rs} -> {r!r} ({type(r).__name__})",
            f"  Got:      {ls} -> {l!r} ({type(l).__name__})"
        ]
        if locs:
            lines.append("  Context:  " + ", ".join(f"{k} = {v!r}" for k, v in locs.items()))
        if isinstance(l, (list, tuple, str)) and isinstance(r, (list, tuple, str)):
            if len(l) != len(r):
                lines.append(f"  Length mismatch: got {len(l)} items, expected {len(r)}")
            if isinstance(l, (list, tuple)) and isinstance(r, (list, tuple)):
                for i, (a, b) in enumerate(zip(l, r)):
                    if a != b:
                        lines.append(f"  First diff at index {i}: got {a!r}, expected {b!r}")
                        break
        raise AssertionError(chr(10) + chr(10).join(lines))

def _check_expr(val, expr):
    __tracebackhide__ = True
    if not val:
        raise AssertionError(chr(10) + f"Assertion Failed: expected {expr} to be truthy, got {val!r}")

try:
    _ip = get_ipython()
    if _ip is not None:
        import ast as _ast

        class _AssertRewriter(_ast.NodeTransformer):
            def __init__(self):
                self._tid = 0

            def visit_Assert(self, node):
                test = node.test
                if isinstance(test, _ast.BoolOp) and isinstance(test.op, _ast.And):
                    res = []
                    for sub in test.values:
                        res.extend(self.visit_Assert(_ast.Assert(test=sub, msg=node.msg)))
                    return res
                if isinstance(test, _ast.Compare) and len(test.ops) == 1:
                    op_map = {_ast.Eq: "==", _ast.NotEq: "!=", _ast.Lt: "<", _ast.LtE: "<=",
                              _ast.Gt: ">", _ast.GtE: ">=", _ast.Is: "is", _ast.IsNot: "is not",
                              _ast.In: "in", _ast.NotIn: "not in"}
                    op_cls = type(test.ops[0])
                    if op_cls in op_map:
                        self._tid += 1
                        la, ra = f"_a_{self._tid}", f"_b_{self._tid}"
                        return [
                            _ast.Assign(targets=[_ast.Name(id=la, ctx=_ast.Store())], value=test.left),
                            _ast.Assign(targets=[_ast.Name(id=ra, ctx=_ast.Store())], value=test.comparators[0]),
                            _ast.Expr(value=_ast.Call(
                                func=_ast.Name(id="_check_cmp", ctx=_ast.Load()),
                                args=[_ast.Name(id=la, ctx=_ast.Load()), _ast.Name(id=ra, ctx=_ast.Load()),
                                      _ast.Constant(value=op_map[op_cls]), _ast.Constant(value=_ast.unparse(test.left)),
                                      _ast.Constant(value=_ast.unparse(test.comparators[0]))],
                                keywords=[]
                            ))
                        ]
                return [_ast.Expr(value=_ast.Call(
                    func=_ast.Name(id="_check_expr", ctx=_ast.Load()),
                    args=[test, _ast.Constant(value=_ast.unparse(test))],
                    keywords=[]
                ))]

        if not any(isinstance(t, _AssertRewriter) for t in _ip.ast_transformers):
            _ip.ast_transformers.append(_AssertRewriter())
except NameError:
    pass

### 16.1 · Two pointers  `Core`

**Task.** Write `pair_sum_sorted(arr, target)` for a sorted list: return the indices `(i, j)` of a pair summing to target, moving a left and a right pointer inward, or `None`.

**Example.** `pair_sum_sorted([1, 3, 4, 6, 9], 10)` -> `(0, 4)`

**Big O.** O(N) time, O(1) space.

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ex-16-1) · [Solution in the answer key](https://bloomcode.github.io/cp/python/answer-key.html#ex-16-1)

In [ ]:
# 16.1 Two pointers
def pair_sum_sorted(arr, target):
    raise NotImplementedError

In [ ]:
def test_ex_16_1():
    assert pair_sum_sorted([1, 3, 4, 6, 9], 10) == (0, 4)
    assert pair_sum_sorted([1, 2], 10) is None


test_ex_16_1()
print('16.1 passed')

### 16.2 · Fixed-size sliding window  `Core`

**Task.** Write `best_window(arr, k)` returning the largest sum of k consecutive values, updating the window sum in O(1) per step.

**Example.** `best_window([2, 1, 5, 1, 3, 2], 3)` -> `9`

**Big O.** O(N) instead of O(N * k).

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ex-16-2) · [Solution in the answer key](https://bloomcode.github.io/cp/python/answer-key.html#ex-16-2)

In [ ]:
# 16.2 Fixed-size sliding window
def best_window(arr, k):
    raise NotImplementedError

In [ ]:
def test_ex_16_2():
    assert best_window([2, 1, 5, 1, 3, 2], 3) == 9


test_ex_16_2()
print('16.2 passed')

### 16.3 · Variable-size sliding window  `Core`

**Task.** Write `longest_unique(s)` returning the length of the longest substring without repeated characters, using a dict of last positions.

**Example.** `longest_unique("abcabcbb")` -> `3`; `longest_unique("pwwkew")` -> `3`

**Big O.** O(N).

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ex-16-3) · [Solution in the answer key](https://bloomcode.github.io/cp/python/answer-key.html#ex-16-3)

In [ ]:
# 16.3 Variable-size sliding window
def longest_unique(s):
    raise NotImplementedError

In [ ]:
def test_ex_16_3():
    assert longest_unique('abcabcbb') == 3
    assert longest_unique('pwwkew') == 3
    assert longest_unique('') == 0
    assert longest_unique('abba') == 2


test_ex_16_3()
print('16.3 passed')

### 16.4 · Prefix sums plus a hash map  `Core`

**Task.** Write `count_subarrays(arr, k)` counting contiguous subarrays that sum to k, storing how often each prefix sum has appeared.

**Example.** `count_subarrays([1, 1, 1], 2)` -> `2`; `count_subarrays([1, 2, 3, -3, 3], 3)` -> `5`

**Big O.** O(N), and it works with negative numbers, unlike a sliding window.

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ex-16-4) · [Solution in the answer key](https://bloomcode.github.io/cp/python/answer-key.html#ex-16-4)

In [ ]:
# 16.4 Prefix sums plus a hash map
def count_subarrays(arr, k):
    raise NotImplementedError

In [ ]:
def test_ex_16_4():
    assert count_subarrays([1, 1, 1], 2) == 2
    assert count_subarrays([1, 2, 3, -3, 3], 3) == 5
    rng = random.Random(1)
    for _ in range(50):
        a = [rng.randint(-3, 3) for _ in range(12)]
        k = rng.randint(-3, 3)
        assert count_subarrays(a, k) == sum((1 for i in range(12) for j in range(i, 12) if sum(a[i:j + 1]) == k))


test_ex_16_4()
print('16.4 passed')

### 16.5 · Grid neighbours: count islands  `Core`

**Task.** Write `count_islands(grid)` counting groups of `#` cells joined up, down, left or right. Loop over a `DIRS` list of `(dr, dc)` pairs and flood-fill with a deque.

**Example.** `count_islands(["##..", "#..#", "...#"])` -> `2`

**Big O.** O(R * C).

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ex-16-5) · [Solution in the answer key](https://bloomcode.github.io/cp/python/answer-key.html#ex-16-5)

In [ ]:
# 16.5 Grid neighbours: count islands
DIRS = [(1, 0), (-1, 0), (0, 1), (0, -1)]


def count_islands(grid):
    raise NotImplementedError

In [ ]:
def test_ex_16_5():
    assert count_islands(['##..', '#..#', '...#']) == 2
    assert count_islands(['...', '...']) == 0


test_ex_16_5()
print('16.5 passed')

### 16.6 · Simulation  `Core`

**Task.** Write `robot(moves)` returning the final `(x, y)` after a string of `U`, `D`, `L`, `R` moves from (0, 0), using a dict from letter to step.

**Example.** `robot("UURDL")` -> `(0, 1)`

**Big O.** O(N).

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ex-16-6) · [Solution in the answer key](https://bloomcode.github.io/cp/python/answer-key.html#ex-16-6)

In [ ]:
# 16.6 Simulation
def robot(moves):
    raise NotImplementedError

In [ ]:
def test_ex_16_6():
    assert robot('UURDL') == (0, 1)
    assert robot('') == (0, 0)


test_ex_16_6()
print('16.6 passed')

### 16.7 · Difference array for range updates  `Core`

**Task.** Write `apply_updates(n, updates)` for updates `(l, r, v)` that add v to every index in [l, r] of a zero array. Mark `diff[l] += v` and `diff[r + 1] -= v`, then take prefix sums.

**Example.** `apply_updates(5, [(1, 3, 2), (2, 4, 3), (0, 2, -1)])` -> `[-1, 1, 4, 5, 3]`

**Big O.** O(N + U) instead of O(N * U).

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ex-16-7) · [Solution in the answer key](https://bloomcode.github.io/cp/python/answer-key.html#ex-16-7)

In [ ]:
# 16.7 Difference array for range updates
def apply_updates(n, updates):
    raise NotImplementedError

In [ ]:
def test_ex_16_7():
    assert apply_updates(5, [(1, 3, 2), (2, 4, 3), (0, 2, -1)]) == [-1, 1, 4, 5, 3]


test_ex_16_7()
print('16.7 passed')

### 16.8 · Greedy: most non-overlapping meetings  `Core`

**Task.** Write `max_meetings(intervals)` for `(start, end)` pairs: sort by end time and take every meeting that starts at or after the last chosen end.

**Example.** `max_meetings([(1, 3), (2, 4), (3, 5), (0, 7), (5, 8)])` -> `3`

**Big O.** O(N log N).

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ex-16-8) · [Solution in the answer key](https://bloomcode.github.io/cp/python/answer-key.html#ex-16-8)

In [ ]:
# 16.8 Greedy: most non-overlapping meetings
def max_meetings(intervals):
    raise NotImplementedError

In [ ]:
def test_ex_16_8():
    assert max_meetings([(1, 3), (2, 4), (3, 5), (0, 7), (5, 8)]) == 3


test_ex_16_8()
print('16.8 passed')

### 16.9 · A full problem, input to output  `Practice`

**Task.** Write `solve(data)`: T test cases, each with N then N integers. For each, print the largest sum of a non-empty contiguous subarray. Use the token iterator from 1.7 and Kadane's running sum.

**Example.** `solve("2\n5\n-2 1 -3 4 -1\n3\n-5 -1 -8")` -> `"4\n-1"`

**Big O.** O(total N). This is the shape most contest solutions take: read, compute per case, join, print.

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ex-16-9) · [Solution in the answer key](https://bloomcode.github.io/cp/python/answer-key.html#ex-16-9)

In [ ]:
# 16.9 A full problem, input to output
def solve(data):
    raise NotImplementedError

In [ ]:
def test_ex_16_9():
    assert solve('2\n5\n-2 1 -3 4 -1\n3\n-5 -1 -8') == '4\n-1'


test_ex_16_9()
print('16.9 passed')